# Phases 0-1: setup, PTB-XL verification, cohorts, splits and cache

Run on Google Colab with Google Drive mounted. Each phase starts only if the previous phase's gate file says GO. See the repository README for the full workflow.

### Cell 0.1: Connect Google Drive and create the project folders

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, shutil, subprocess, datetime, hashlib, zipfile, ast
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/ecg_leak")
folders = ["src/ecgleak", "notebooks", "data/raw/ptbxl-1.0.3", "data/interim", "data/splits",
           "data/cache", "runs", "results/audit", "results/figures", "gates"]
for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
(PROJECT / "src/ecgleak/__init__.py").touch()
print("Folders ready at", PROJECT)

### Cell 0.2: Install libraries

In [ ]:
!pip install -q wfdb neurokit2 pyyaml

### Cell 0.3: Write config.yaml

In [ ]:
CONFIG_TEXT = """\
data:
  version: 1.0.3
  raw_dir: /content/drive/MyDrive/PTB-XL
  local_dir: /content/ptbxl
  fs: 500
  record_seconds: 10
  lead: I
  expected_records: 21799
  expected_patients: 18869
qc:
  flat_std_mv: 0.01
  min_beats: 4
  max_beats: 35
cohorts:
  min_records_identity: 2
  identity_train_frac: 0.80
  lockbox_frac: 0.30
  n_folds: 5
  min_test_per_class: 300
preprocess:
  band_hz: [0.5, 40]
  filter_order: 4
  window_s: 2.5
  beat_ms: [-250, 400]
model:
  emb_dim: 256
  cnn_channels: [32, 64, 128]
  kernel: 7
  transformer_layers: 2
  heads: 4
arcface: {s: 30.0, m: 0.50}
train: {epochs: 60, batch: 128, lr: 1.0e-3, weight_decay: 1.0e-4, warmup_frac: 0.05, patience: 10}
leakage:
  n_untrained: 5
  untrained_seeds: [100, 101, 102, 103, 104]
  tasks: [CD, MI, STTC, HYP]
  primary: CD
  C_grid: [0.001, 0.01, 0.1, 1, 10, 100]
stats: {n_boot: 2000, n_perm: 1000, alpha: 0.05, ci_halfwidth_target: 0.03}
seeds: [0, 1, 2]
"""
cfg_path = PROJECT / "config.yaml"
cfg_path.write_text(CONFIG_TEXT)
print("Wrote", cfg_path)

### Cell 0.4: Shared module: config.py

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/config.py
import datetime, json, os, platform, random
import numpy as np, yaml

DIRS = {'interim': 'data/interim', 'splits': 'data/splits',
        'cache': 'data/cache', 'runs': 'runs', 'results': 'results', 'audit': 'results/audit',
        'figures': 'results/figures', 'gates': 'gates'}

def set_seeds(seed, deterministic=False):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

def environment():
    env = {'python': platform.python_version(), 'numpy': np.__version__,
           'time': datetime.datetime.now().isoformat(timespec='seconds')}
    try:
        import torch
        env.update(torch=torch.__version__, cuda=torch.cuda.is_available(),
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    except ImportError:
        env['torch'] = None
    return env

def load(path, seed=None, verbose=True):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    root = os.path.dirname(os.path.abspath(path))
    cfg['paths'] = {'root': root, 'raw': cfg['data']['raw_dir'], 'local_data': cfg['data']['local_dir'],
                    **{k: os.path.join(root, v) for k, v in DIRS.items()}}
    cfg['seed'] = cfg['seeds'][0] if seed is None else seed
    set_seeds(cfg['seed'])
    if verbose:
        e = environment()
        print(f"Root {root} | seed {cfg['seed']} | Python {e['python']} | torch {e.get('torch')} | GPU {e.get('gpu') or 'none (CPU)'}")
    return cfg

def freeze(cfg, run_dir):
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, 'config.frozen.yaml'), 'w') as f:
        yaml.safe_dump({k: v for k, v in cfg.items() if k != 'paths'}, f, sort_keys=False)
    with open(os.path.join(run_dir, 'environment.json'), 'w') as f:
        json.dump(environment(), f, indent=2)

### Cell 0.5: Load config, check the PTB-XL upload, Phase 0 gate

In [ ]:
sys.path.insert(0, str(PROJECT / "src"))
import importlib, ecgleak.config as ecfg
importlib.reload(ecfg)

CFG = ecfg.load(str(PROJECT / "config.yaml"))
PATHS = CFG["paths"]
LOCAL = CFG["data"]["local_dir"].rstrip("/") + "/"
os.makedirs(LOCAL, exist_ok=True)

def free_gb(p):
    return shutil.disk_usage(p).free / 1e9

RAW = PATHS["raw"].rstrip("/") + "/"
checks0 = {"Drive project folder": PROJECT.is_dir(),
           "Config is PTB-XL 1.0.3": str(CFG["data"]["version"]) == "1.0.3",
           "Colab disk >= 6 GB free": free_gb(LOCAL) >= 6,
           "PTB-XL files on Drive": all(os.path.exists(RAW + f) for f in ["records500", "ptbxl_database.csv", "scp_statements.csv", "SHA256SUMS.txt"])}
for name, ok in checks0.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
assert all(checks0.values()), "Phase 0 gate failed: fix the FAIL lines above"
print("PHASE 0: GO")

### Cell 1.1: Copy records500 and CSVs from Drive to the Colab disk

In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
WAVE_DIR, FILE_COL = "records500", "filename_hr"
KEEP = ("ptbxl_database.csv", "scp_statements.csv", "SHA256SUMS.txt", "LICENSE.txt", "RECORDS")
RAW = PATHS["raw"].rstrip("/") + "/"
P = LOCAL

def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

for name in KEEP:
    if os.path.isfile(RAW + name):
        shutil.copyfile(RAW + name, P + name)

jobs = []
for sub in sorted(os.listdir(RAW + WAVE_DIR)):
    if os.path.isdir(RAW + WAVE_DIR + "/" + sub):
        os.makedirs(P + WAVE_DIR + "/" + sub, exist_ok=True)
        jobs += [f"{WAVE_DIR}/{sub}/{f}" for f in os.listdir(RAW + WAVE_DIR + "/" + sub)]
todo = [r for r in jobs if not (os.path.isfile(P + r) and os.path.getsize(P + r) == os.path.getsize(RAW + r))]
print(f"Files in Drive records500: {len(jobs):,} | still to copy: {len(todo):,}")

def copy_one(rel):
    shutil.copyfile(RAW + rel, P + rel)

with ThreadPoolExecutor(max_workers=32) as pool:            # parallel: Drive is slow per file, fast in bulk
    list(tqdm(pool.map(copy_one, todo), total=len(todo), desc="Copying to Colab disk", mininterval=10))
print("Ready on the Colab disk:", P)
SOURCE = RAW

### Cell 1.2: Verify every file against PhysioNet SHA-256

In [ ]:
import pandas as pd, numpy as np
published = {}
for line in open(P + "SHA256SUMS.txt"):
    parts = line.strip().split(maxsplit=1)
    if len(parts) == 2:
        published[parts[1].lstrip("*").removeprefix("./")] = parts[0].lower()
targets = [r for r in published if r in KEEP[:2] or r.startswith(WAVE_DIR + "/")]
status = pd.Series({r: ("missing" if not os.path.isfile(P + r) else
                        ("ok" if sha256(P + r) == published[r] else "MISMATCH"))
                    for r in tqdm(targets, desc="SHA-256", mininterval=10)}, dtype=object)
print(status.value_counts().to_string())
status[status != "ok"].to_csv(f"{PATHS['audit']}/phase1_checksum_problems.csv")
CHECKSUMS_OK = len(status) > 0 and bool((status == "ok").all())
if not CHECKSUMS_OK:
    print("Problem files are listed in results/audit/phase1_checksum_problems.csv")

### Cell 1.3: Version 1.0.3, completeness, columns

In [ ]:
db = pd.read_csv(P + "ptbxl_database.csv", index_col="ecg_id")
N_REC, N_PAT = len(db), db["patient_id"].nunique()
VERSION_OK = (N_REC, N_PAT) == (CFG["data"]["expected_records"], CFG["data"]["expected_patients"])
print(f"Records {N_REC:,} | Patients {N_PAT:,} | matches 1.0.3: {VERSION_OK}")

missing = [s + e for s in db[FILE_COL] for e in (".dat", ".hea") if not os.path.isfile(P + s + e)]
COMPLETE = not missing
print("Missing waveform files:", len(missing))

REQUIRED = ["patient_id", "recording_date", "scp_codes", "age", "sex", FILE_COL]
COLUMNS_OK = all(c in db.columns for c in REQUIRED)
print("Required columns present:", COLUMNS_OK)

### Cell 1.4: Read lead I from every record

In [ ]:
import wfdb
FS = CFG["data"]["fs"]
N_SAMP = FS * CFG["data"]["record_seconds"]

lead_I = np.full((N_REC, N_SAMP), np.nan, dtype=np.float32)
read_err = {}
for i, (ecg_id, stem) in enumerate(tqdm(db[FILE_COL].items(), total=N_REC, desc="Reading lead I", mininterval=10)):
    try:
        rec = wfdb.rdrecord(P + stem, channel_names=["I"])
        if rec.fs == FS and rec.sig_len == N_SAMP:
            lead_I[i] = rec.p_signal[:, 0]
        else:
            read_err[ecg_id] = f"fs={rec.fs}, len={rec.sig_len}"
    except Exception as e:
        read_err[ecg_id] = type(e).__name__
print(f"Read {N_REC - len(read_err):,} records | problems: {len(read_err)}")

### Cell 1.5: Diagnostic superclass labels

In [ ]:
SUPER = ["NORM", "MI", "STTC", "CD", "HYP"]
scp = pd.read_csv(P + "scp_statements.csv", index_col=0)
diag_class = scp.loc[scp["diagnostic"] == 1, "diagnostic_class"]

def superclasses(codes_text):
    codes = ast.literal_eval(codes_text)
    return {diag_class[c] for c in codes if c in diag_class.index}

sets = db["scp_codes"].map(superclasses)
labels = pd.DataFrame({c: sets.map(lambda s, c=c: int(c in s)) for c in SUPER}, index=db.index)
labels.to_csv(f"{PATHS['interim']}/labels_superclass.csv")
print("Records per superclass:\n" + labels.sum().to_string())
print("Records with no diagnostic superclass:", int((labels.sum(axis=1) == 0).sum()))

### Cell 1.6: Filter, R-peaks, median beats, quality control

In [ ]:
import neurokit2 as nk
from scipy.signal import butter, sosfiltfilt

lo, hi = CFG["preprocess"]["band_hz"]
sos = butter(CFG["preprocess"]["filter_order"], [lo, hi], btype="band", fs=FS, output="sos")
b0, b1 = [int(round(ms * FS / 1000)) for ms in CFG["preprocess"]["beat_ms"]]   # -125, 200 samples
BEAT_LEN = b1 - b0

ok_read = np.isfinite(lead_I).all(axis=1)
filtered = np.zeros_like(lead_I)
filtered[ok_read] = sosfiltfilt(sos, lead_I[ok_read], axis=1)
raw_std = np.nanstd(lead_I, axis=1)

median_beats = np.full((N_REC, BEAT_LEN), np.nan, dtype=np.float32)
MAX_PEAKS = 40
rpeaks = np.full((N_REC, MAX_PEAKS), -1, dtype=np.int32)
n_beats = np.zeros(N_REC, dtype=int)
for i in tqdm(np.where(ok_read & (raw_std > CFG["qc"]["flat_std_mv"]))[0], desc="R-peaks", mininterval=10):
    try:
        _, info = nk.ecg_peaks(filtered[i], sampling_rate=FS)
        pk = np.asarray(info["ECG_R_Peaks"], dtype=int)
    except Exception:
        continue
    n_beats[i] = len(pk)
    rpeaks[i, :min(len(pk), MAX_PEAKS)] = pk[:MAX_PEAKS]
    full = pk[(pk + b0 >= 0) & (pk + b1 <= N_SAMP)]
    if len(full):
        median_beats[i] = np.median(np.stack([filtered[i, p + b0:p + b1] for p in full]), axis=0)

std_f = filtered.std(axis=1, keepdims=True)
signals = ((filtered - filtered.mean(axis=1, keepdims=True)) / np.where(std_f > 0, std_f, 1)).astype(np.float16)

reason = pd.Series("", index=db.index, dtype=object)
reason[~ok_read] = "unreadable"
reason[(reason == "") & (raw_std <= CFG["qc"]["flat_std_mv"])] = "flat lead I"
reason[(reason == "") & ((n_beats < CFG["qc"]["min_beats"]) | (n_beats > CFG["qc"]["max_beats"]))] = "implausible beat count"
reason[(reason == "") & ~np.isfinite(median_beats).all(axis=1)] = "no full beat"
QC_PASS = (reason == "").to_numpy()
reason[reason != ""].rename("reason").to_csv(f"{PATHS['audit']}/phase1_excluded.csv")
print(f"QC pass: {QC_PASS.sum():,} / {N_REC:,}")
print(reason[reason != ""].value_counts().to_string())

### Cell 1.7: Cohorts and patient-disjoint splits

In [ ]:
rng = np.random.default_rng(CFG["seed"])
meta = db[["patient_id", "recording_date", "age", "sex"]].copy()
meta["patient_id"] = meta["patient_id"].astype(int)
meta["qc_pass"] = QC_PASS
per_patient = meta.groupby("patient_id").size()          # counted before QC, so cohorts never shift with QC

id_pats = np.array(sorted(per_patient[per_patient >= CFG["cohorts"]["min_records_identity"]].index))
probe_pats = np.array(sorted(per_patient[per_patient == 1].index))

rng.shuffle(id_pats)
n_train = int(round(CFG["cohorts"]["identity_train_frac"] * len(id_pats)))
split_of = {p: "id_train" for p in id_pats[:n_train]}
split_of.update({p: "id_eval" for p in id_pats[n_train:]})

# Probe cohort: stratify on the label pattern, then 30% lockbox and 5 folds in probe-dev.
probe_meta = meta[meta["patient_id"].isin(probe_pats)].join(labels)
key = probe_meta[SUPER].astype(str).agg("".join, axis=1)
counts = key.value_counts()
key = key.where(key.map(counts) >= 10, "rare")
fold_of = {}
for _, grp in probe_meta.groupby(key):
    pats = grp["patient_id"].to_numpy().copy()
    rng.shuffle(pats)
    n_lock = int(round(CFG["cohorts"]["lockbox_frac"] * len(pats)))
    for p in pats[:n_lock]:
        split_of[p] = "lockbox"
    for j, p in enumerate(pats[n_lock:]):
        split_of[p] = "probe_dev"
        fold_of[p] = j % CFG["cohorts"]["n_folds"]

meta["split"] = meta["patient_id"].map(split_of)
meta["fold"] = meta["patient_id"].map(fold_of).fillna(-1).astype(int)
splits = meta.join(labels)
splits.to_csv(f"{PATHS['splits']}/splits.csv")
SPLITS_SHA = sha256(f"{PATHS['splits']}/splits.csv")

pats_by_split = {s: set(g["patient_id"]) for s, g in splits.groupby("split")}
names = list(pats_by_split)
OVERLAP = sum(len(pats_by_split[a] & pats_by_split[b]) for i, a in enumerate(names) for b in names[i + 1:])
used = splits[splits["qc_pass"]]
print(used.groupby("split").agg(records=("patient_id", "size"), patients=("patient_id", "nunique")).to_string())
print("Patients in more than one split:", OVERLAP)

def task_counts(df, cls):
    pos = df[cls] == 1
    neg = (df["NORM"] == 1) & (df[SUPER].sum(axis=1) == 1)
    return int(pos.sum()), int(neg.sum())

lock = used[used["split"] == "lockbox"]
task_table = pd.DataFrame({c: task_counts(lock, c) for c in CFG["leakage"]["tasks"]}, index=["positive", "NORM-only"]).T
print("\nLockbox patients per task:\n" + task_table.to_string())
task_table.to_csv(f"{PATHS['audit']}/phase1_lockbox_task_counts.csv")
PRIMARY_OK = min(task_counts(lock, CFG["leakage"]["primary"])) >= CFG["cohorts"]["min_test_per_class"]

### Cell 1.8: Diagnosis persistence across visits (H5)

In [ ]:
idc = splits[splits["split"].isin(["id_train", "id_eval"]) & splits["qc_pass"]].copy()
idc["date"] = pd.to_datetime(idc["recording_date"], errors="coerce")
idc = idc.sort_values(["patient_id", "date"])
nxt = idc.groupby("patient_id")[SUPER].shift(-1)
pairs = idc[SUPER][nxt.notna().all(axis=1)]
nxt = nxt.dropna()
persist = pd.DataFrame({c: {"visit-1 positives": int(pairs[c].sum()),
                            "still positive at next visit": int(((pairs[c] == 1) & (nxt[c] == 1)).sum())}
                        for c in SUPER}).T
persist["persistence"] = (persist["still positive at next visit"] / persist["visit-1 positives"].replace(0, np.nan)).round(3)
persist.to_csv(f"{PATHS['interim']}/persistence.csv")
print(f"Consecutive visit pairs: {len(pairs):,}\n" + persist.to_string())

### Cell 1.9: Save the heartbeat cache

In [ ]:
CACHE = PATHS["cache"]
age = db["age"].to_numpy(dtype=float)
sex = db["sex"].to_numpy(dtype=float)
split_arr = splits["split"].to_numpy()
cache_files = {}
for s in ["id_train", "id_eval", "probe_dev", "lockbox"]:
    m = (split_arr == s) & QC_PASS
    path = f"{CACHE}/{s}.npz"
    np.savez(path, ecg_id=db.index.to_numpy()[m], patient_id=splits["patient_id"].to_numpy()[m],
             fold=splits["fold"].to_numpy()[m], signal=signals[m], rpeaks=rpeaks[m],
             median_beat=median_beats[m], labels=labels.to_numpy()[m], label_names=np.array(SUPER),
             age=age[m], sex=sex[m])
    cache_files[s] = {"records": int(m.sum()), "sha256": sha256(path)}
    print(f"{s:10s} {int(m.sum()):6,} records -> {path}")

import matplotlib.pyplot as plt
i = int(np.where(QC_PASS)[0][0])
t = np.arange(BEAT_LEN) / FS * 1000 + CFG["preprocess"]["beat_ms"][0]
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(np.arange(N_SAMP) / FS, signals[i].astype(float), lw=0.7)
pk = rpeaks[i][rpeaks[i] >= 0]
ax[0].plot(pk / FS, signals[i][pk].astype(float), "rx")
ax[0].set_title(f"Filtered, z-scored lead I with R-peaks (ecg_id {db.index[i]})"); ax[0].set_xlabel("Time (s)")
ax[1].plot(t, median_beats[i]); ax[1].set_title("Median beat"); ax[1].set_xlabel("ms from R-peak")
plt.tight_layout(); plt.savefig(f"{PATHS['figures']}/phase1_preprocessing_example.png", dpi=150); plt.show()

### Cell 1.10: Phase 1 gate

In [ ]:
gate = {"Checksums (all files)": CHECKSUMS_OK, "Version 1.0.3": VERSION_OK, "Completeness": COMPLETE,
        "Metadata columns": COLUMNS_OK, "No patient in two splits": OVERLAP == 0,
        f"Lockbox >= {CFG['cohorts']['min_test_per_class']} per class ({CFG['leakage']['primary']} vs NORM)": PRIMARY_OK,
        "Cache written": all(v["records"] > 0 for v in cache_files.values())}
GO = all(gate.values())
print("PHASE 1 GATE\n" + "-" * 50)
for k, v in gate.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("-" * 50)
print("GO: Phase 1 complete. Next: Phase 2 (baselines + pilot)." if GO else "NO-GO: fix the FAIL lines above.")
json.dump({"phase": 1, "go": GO, "checks": gate, "records": N_REC, "patients": N_PAT,
           "qc_pass": int(QC_PASS.sum()), "source": SOURCE, "splits_sha256": SPLITS_SHA,
           "cache": cache_files, "environment": ecfg.environment()},
          open(f"{PATHS['gates']}/gate_1.json", "w"), indent=2, default=str)